# FYP2 Model Training - AutoDL A100-PCIE-40GB

**Usage**: run each cell from top to bottom.

- Dataset path: `/root/autodl-tmp/CCV2.v2i.yolov81024/`
- Project code path: `/root/autodl-tmp/fyp_withoutRebar/`
- Training outputs are written to `/root/autodl-tmp/fyp_withoutRebar/runs/train/`

## 0. Environment Check

In [ ]:
import subprocess, sys, os

print("Python:", sys.version)
print("Working dir:", os.getcwd())

# GPU check
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
print("GPU:", r.stdout.strip())

# Disk check
for path in ["/root/autodl-tmp"]:
    if os.path.exists(path):
        s = os.statvfs(path)
        free_gb = s.f_frsize * s.f_bavail / 1e9
        print(f"{path}: {free_gb:.0f} GB free")

## 1. Install Dependencies

In [ ]:
# AutoDL images usually include PyTorch and CUDA. Install the remaining dependencies.
!pip install "numpy<2" ultralytics opencv-python-headless scikit-image flask -q

# Verify
import torch, ultralytics
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
print(f"Ultralytics {ultralytics.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Check Project Code and Dataset

> Before running, upload the following files to `/root/autodl-tmp/` using AutoDL upload or `scp`:
> - `fyp_withoutRebar/`（project code, or extracted project archive）
> - `CCV2.v2i.yolov81024/`（dataset）
>
> AutoDL Jupyter supports upload by dragging files into the left file browser.

In [ ]:
import os

BASE = "/root/autodl-tmp"
PROJECT = os.path.join(BASE, "fyp_withoutRebar")
DATA = os.path.join(BASE, "CCV2.v2i.yolov81024")

print("Project dir:", "OK" if os.path.isdir(PROJECT) else "MISSING — please upload fyp_withoutRebar/")
print("Dataset dir:", "OK" if os.path.isdir(DATA) else "MISSING — please upload CCV2.v2i.yolov81024/")

# Check key files
for f in ["train.py", "modules/skeleton.py", "requirements.txt"]:
    p = os.path.join(PROJECT, f)
    print(f"  {f}: {'OK' if os.path.exists(p) else 'MISSING'}")

for f in ["data.yaml", "train/images", "test/images"]:
    p = os.path.join(DATA, f)
    print(f"  data/{f}: {'OK' if os.path.exists(p) else 'MISSING'}")

## 3. Dataset Audit

In [ ]:
import sys; sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

from pathlib import Path
from dataset_audit import audit_dataset
import json

data_yaml = Path(os.path.join(DATA, "data.yaml"))
audit = audit_dataset(data_yaml)

print(json.dumps({
    "total_images": sum(s["images"] for s in audit["splits"].values()),
    "splits": {k: f"{v['images']} images, {sum(v['instances'].values())} instances"
               for k, v in audit["splits"].items()},
    "missing_anything": any(
        v.get("missing_labels") or v.get("missing_images")
        for v in audit["splits"].values()
    )
}, indent=2))

## 4. Training Configuration

**A100 40GB note**：batch size can often be increased above 12 for faster convergence.

Experiments are ordered by priority:

In [ ]:
# ═══════════════════════════════════════════════
# Experiment configurations
# Adjust name, epochs, and batch as needed
# ═══════════════════════════════════════════════

EXPERIMENTS = [
    {
        "name": "ccv2_yolov8x_1024_balanced_200ep",
        "model": "yolov8x-seg.pt",
        "epochs": 200,
        "batch": 20,
        "balance": True,
        "description": "E1: YOLOv8x + balanced oversampling, 200 epochs (highest priority)",
    },
    {
        "name": "ccv2_yolo11x_1024_balanced_200ep",
        "model": "yolo11x-seg.pt",
        "epochs": 200,
        "batch": 16,
        "balance": True,
        "description": "E2: YOLO11x + balanced, architecture comparison",
    },
    {
        "name": "ccv2_yolov8x_1024_baseline_200ep",
        "model": "yolov8x-seg.pt",
        "epochs": 200,
        "batch": 20,
        "balance": False,
        "description": "E3: YOLOv8x imbalanced baseline, 200 epochs (ablation control)",
    },
]

print("Planned training experiments:")
for i, e in enumerate(EXPERIMENTS):
    bal = "balanced" if e["balance"] else "baseline"
    print(f"  {i+1}. {e['name']} ({e['model']}, {e['epochs']}ep, batch={e['batch']}, {bal})")

## 5. Start Training

In [ ]:
import subprocess, time, json
from datetime import datetime

DATA_YAML = os.path.join(DATA, "data.yaml")
LOG = []

for idx, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(EXPERIMENTS)}] {exp['description']}")
    print(f"{'='*60}")
    start = time.time()
    
    cmd = [
        sys.executable, "train.py",
        "--model", exp["model"],
        "--data", DATA_YAML,
        "--device", "0",
        "--imgsz", "1024",
        "--batch", str(exp["batch"]),
        "--epochs", str(exp["epochs"]),
        "--name", exp["name"],
    ]
    if exp["balance"]:
        cmd += ["--balance", "--minority-classes", "1", "2", "--minority-copies", "3"]
    
    print("Command:", " ".join(cmd))
    
    try:
        result = subprocess.run(cmd, cwd=PROJECT, check=True)
        elapsed_m = (time.time() - start) / 60
        LOG.append({"name": exp["name"], "status": "OK", "elapsed_min": round(elapsed_m, 1)})
        print(f"DONE ({elapsed_m:.1f} min)")
    except subprocess.CalledProcessError as e:
        elapsed_m = (time.time() - start) / 60
        LOG.append({"name": exp["name"], "status": "FAILED", "elapsed_min": round(elapsed_m, 1)})
        print(f"FAILED ({elapsed_m:.1f} min), continuing to next run...")

print(f"\n{'='*60}")
print("Training summary")
print(f"{'='*60}")
for entry in LOG:
    status_icon = "✓" if entry["status"] == "OK" else "✗"
    print(f"  {status_icon} {entry['name']} ({entry['elapsed_min']} min) - {entry['status']}")

## 6. Evaluate Best Models

Run formal evaluation for each successful training experiment.

In [ ]:
TRAIN_DIR = os.path.join(PROJECT, "runs", "train")

for exp in EXPERIMENTS:
    best_pt = os.path.join(TRAIN_DIR, exp["name"], "weights", "best.pt")
    if not os.path.exists(best_pt):
        print(f"SKIP {exp['name']} — best.pt not found")
        continue
    
    print(f"\n{'='*60}")
    print(f"Evaluating: {exp['name']}")
    
    for imgsz in [640, 1024]:
        output = os.path.join(PROJECT, "runs", "evaluate", f"{exp['name']}_{imgsz}")
        cmd = [
            sys.executable, "evaluate.py",
            "--weights", best_pt,
            "--data", DATA_YAML,
            "--split", "test",
            "--imgsz", str(imgsz),
            "--output", output,
            "--save-visuals",
        ]
        print(f"  imgsz={imgsz} ...")
        subprocess.run(cmd, cwd=PROJECT)
        
        # Print key metrics
        mf = os.path.join(output, "metrics.json")
        if os.path.exists(mf):
            with open(mf) as f:
                m = json.load(f)
            print(f"    Box  mAP50: {m['map']['box']['map50']:.3f}  |  mAP50-95: {m['map']['box']['map']:.3f}")
            print(f"    Mask mAP50: {m['map']['mask']['map50']:.3f}  |  mAP50-95: {m['map']['mask']['map']:.3f}")
            print(f"    Per-image count MAE: {m['counts']['per_image_total']['mae']:.3f}")
            print(f"    Inference: {m['counts']['latency']['inference']['mean_ms']:.0f} ms")

## 7. Comparison Summary Table

In [ ]:
print(f"{'Experiment':<45} {'Box mAP50':>10} {'Mask mAP50':>10} {'Count MAE':>10} {'Latency':>10}")
print("-" * 90)

for exp in EXPERIMENTS:
    for imgsz in [640, 1024]:
        mf = os.path.join(PROJECT, "runs", "evaluate", f"{exp['name']}_{imgsz}", "metrics.json")
        if not os.path.exists(mf):
            continue
        with open(mf) as f:
            m = json.load(f)
        box = m['map']['box']['map50']
        mask = m['map']['mask']['map50']
        mae = m['counts']['per_image_total']['mae']
        lat = m['counts']['latency']['inference']['mean_ms']
        label = f"{exp['name'][:35]}@{imgsz}"
        print(f"{label:<45} {box:10.3f} {mask:10.3f} {mae:10.3f} {lat:8.0f} ms")

## 8. Package Outputs

Run this cell after training to create `/root/autodl-tmp/results.tar.gz` with all selected outputs.
Download the archive from the AutoDL file browser.

In [ ]:
import tarfile, glob

OUT = "/root/autodl-tmp/results.tar.gz"

with tarfile.open(OUT, "w:gz") as tar:
    # best.pt for each experiment
    for exp in EXPERIMENTS:
        best = os.path.join(PROJECT, "runs", "train", exp["name"], "weights", "best.pt")
        if os.path.exists(best):
            tar.add(best, arcname=f"{exp['name']}/best.pt")
        # metrics.json
        for mname in ["metrics.json", "run_config.json"]:
            mp = os.path.join(PROJECT, "runs", "train", exp["name"], mname)
            if os.path.exists(mp):
                tar.add(mp, arcname=f"{exp['name']}/{mname}")
    
    # Evaluation outputs
    for f in glob.glob(os.path.join(PROJECT, "runs", "evaluate", "*", "metrics.json")):
        rel = os.path.relpath(f, os.path.join(PROJECT, "runs", "evaluate"))
        tar.add(f, arcname=f"evaluate/{rel}")

size_mb = os.path.getsize(OUT) / 1e6
print(f"Package created: {OUT} ({size_mb:.1f} MB)")
print("Find results.tar.gz in the left file browser and download it")
print(f"\nAfter downloading, extract locally with: tar -xzf results.tar.gz")

---

## Notes

- **Stopping early**: use the Jupyter stop button. Re-run Step 5 after adjusting the `EXPERIMENTS` list if needed.
- **TensorBoard**: run `tensorboard --logdir runs/train --port 6006` in the AutoDL terminal and view it through port forwarding.
- **Shutdown reminder**: stop the cloud instance after training to avoid unnecessary charges.
- **If A100 batch=20 causes OOM**: reduce the batch size to 16 or 12.